In [3]:
!nvidia-smi

Wed Sep 16 06:58:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!apt-get update -qq
!apt-get install -y zstd pciutils
!pip -q install pandas openpyxl requests

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils zstd
0 upgraded, 4 newly installed, 0 to remove and 85 not upgraded.
Need to get 1,025 kB of archives.
After this operation, 3,604 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 pci.ids all 0.0~2024.03.31-1ubuntu0.1 [275 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libpci3 amd64 1:3.

In [6]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

response = requests.get(
    "http://127.0.0.1:11434/api/tags"
)

print("Ollama status:", response.status_code)

Ollama status: 200


In [8]:
!ollama pull medgemma1.5:4b

In [9]:
!ollama list

NAME              ID              SIZE      MODIFIED               
medgemma1.5:4b    433252621ab1    3.3 GB    Less than a second ago    


In [10]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path(
    "/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_LLM_Benchmark.xlsx"
)

RESULT_FILE = Path(
    "/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_MedGemma_Results.xlsx"
)

print("Input exists:", INPUT_FILE.exists())

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Benchmark"
)

print("Number of records:", len(df))

Input exists: True
Number of records: 500


In [11]:
df["medgemma_output"] = ""
df["medgemma_raw"] = ""
df["medgemma_status"] = ""
df["medgemma_runtime_sec"] = ""

df.to_excel(
    RESULT_FILE,
    index=False
)

print("Created:")
print(RESULT_FILE)

Created:
/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_MedGemma_Results.xlsx


In [12]:
import json
import time
import requests

MODEL = "medgemma1.5:4b"

URL = "http://127.0.0.1:11434/api/chat"


SYSTEM_MESSAGE = """
You are a biomedical information extraction system.
Follow the user's extraction instructions exactly.
Return only the requested JSON output.
""".strip()


EXTRACTION_TEMPLATE = """
Extract ALL diseases, disorders, syndromes, symptoms, or clinical conditions
for which the drug is explicitly indicated in the following
Indications and Usage text.

Rules:

1. Read the entire supplied text.
2. Include every explicitly stated clinical indication.
3. Include symptoms or clinical conditions when explicitly indicated
   for treatment, prevention, management, or relief.
4. Preserve clinically meaningful disease subtypes and modifiers.
5. Use wording from the supplied text as closely as possible.
6. Do not intentionally normalize or generalize disease names.
7. Do not infer conditions that are not explicitly supported by the text.
8. Exclude conditions explicitly stated as NOT indicated or NOT recommended.
9. Exclude causative organisms when they merely describe the cause of infection.
10. Exclude mechanisms, procedures, dosing information,
    treatment regimens, and patient demographics.
11. Remove duplicate clinical conditions.

Return ONLY valid JSON:

{{"diseases":["condition 1","condition 2"]}}

If there are genuinely no indications:

{{"diseases":[]}}

INDICATIONS AND USAGE:

{indication_text}
""".strip()

In [13]:
def parse_output(raw_output):

    raw_output = str(raw_output).strip()

    try:

        parsed = json.loads(
            raw_output
        )

    except json.JSONDecodeError:

        start = raw_output.find("{")
        end = raw_output.rfind("}")

        if start == -1 or end == -1:

            raise ValueError(
                "No JSON object found in model response."
            )

        parsed = json.loads(
            raw_output[start:end + 1]
        )


    if "diseases" not in parsed:

        raise ValueError(
            "JSON does not contain 'diseases'."
        )


    diseases = parsed["diseases"]

    if not isinstance(diseases, list):

        raise ValueError(
            "'diseases' must be a list."
        )


    cleaned = []
    seen = set()


    for disease in diseases:

        if not isinstance(disease, str):
            continue

        disease = disease.strip()

        if not disease:
            continue

        key = disease.lower()

        if key not in seen:

            seen.add(key)
            cleaned.append(disease)


    return cleaned

In [14]:
def extract_medgemma(indication_text):

    user_message = EXTRACTION_TEMPLATE.format(
        indication_text=indication_text
    )

    payload = {

        "model": MODEL,

        "messages": [

            {
                "role": "system",
                "content": SYSTEM_MESSAGE
            },

            {
                "role": "user",
                "content": user_message
            }
        ],

        "stream": False,

        "options": {

            "temperature": 0.1,

            "num_ctx": 4096,

            "num_predict": 3000
        }
    }


    start_time = time.time()


    response = requests.post(
        URL,
        json=payload,
        timeout=240
    )

    response.raise_for_status()


    raw = (
        response.json()
        ["message"]
        ["content"]
    )


    runtime = round(
        time.time() - start_time,
        2
    )


    # Preserve raw output even if JSON parsing fails
    try:

        diseases = parse_output(raw)

        status = (
            "OK"
            if diseases
            else "OK_EMPTY"
        )

    except Exception:

        diseases = []
        status = "FORMAT_FAILED"


    return diseases, raw, runtime, status

In [15]:
test_text = """
Metoprolol is indicated for the treatment of hypertension and angina pectoris.
"""

diseases, raw, runtime, status = extract_medgemma(
    test_text
)

print("RAW:")
print(raw)

print("\nPARSED:")
print(diseases)

print("\nSTATUS:")
print(status)

print("\nRuntime:", runtime)

RAW:
<unused94> thought
The user wants me to extract all diseases, disorders, syndromes, symptoms, or clinical conditions mentioned in the "Indications and Usage" section of the provided text.

1.  **Read the text:** "Metoprolol is indicated for the treatment of hypertension and angina pectoris."
2.  **Identify potential conditions:** "hypertension" and "angina pectoris".
3.  **Check against rules:**
    *   Are these explicitly stated? Yes.
    *   Are they diseases, disorders, syndromes, symptoms, or clinical conditions? Yes, hypertension is a disorder, and angina pectoris is a clinical condition.
    *   Are there clinically meaningful subtypes/modifiers? No, these are the basic terms.
    *   Is the wording close to the text? Yes.
    *   Are they normalized/generalized? No.
    *   Are they inferred? No.
    *   Are conditions explicitly excluded? No.
    *   Are causative organisms excluded? No.
    *   Are mechanisms/procedures/dosing excluded? No.
    *   Are duplicates removed

In [16]:
for i in range(5):

    record_id = df.loc[i, "record_id"]

    indication_text = str(
        df.loc[i, "indication_text"]
    )


    print()
    print("=" * 70)
    print(f"RECORD {i + 1}/5")
    print("Record ID:", record_id)

    print()
    print("EXPERT GOLD:")
    print(
        df.loc[i, "gold_expert_mentions"]
    )


    try:

        diseases, raw, runtime, status = extract_medgemma(
            indication_text
        )


        df.at[
            i,
            "medgemma_output"
        ] = json.dumps(
            diseases,
            ensure_ascii=False
        )


        df.at[
            i,
            "medgemma_raw"
        ] = raw


        df.at[
            i,
            "medgemma_status"
        ] = status


        df.at[
            i,
            "medgemma_runtime_sec"
        ] = runtime


        print()
        print("MEDGEMMA OUTPUT:")
        print(diseases)

        print(
            "Status:",
            status
        )

        print(
            "Runtime:",
            runtime,
            "seconds"
        )


    except Exception as e:

        df.at[
            i,
            "medgemma_status"
        ] = "FAILED"

        df.at[
            i,
            "medgemma_raw"
        ] = str(e)

        print(
            "TECHNICAL ERROR:",
            e
        )


df.to_excel(
    RESULT_FILE,
    index=False
)

print()
print("Saved:")
print(RESULT_FILE)


RECORD 1/5
Record ID: 10004

EXPERT GOLD:
Erosive Esophagitis; Gastroesophageal Reflux Disease; GERD; Zollinger Ellison Syndrome

MEDGEMMA OUTPUT:
[]
Status: FORMAT_FAILED
Runtime: 12.68 seconds

RECORD 2/5
Record ID: 10008

EXPERT GOLD:
Anxiety Disorders; anxiety disorder; Anxiety; Anxiety associated with depression; Panic Disorder; agoraphobia

MEDGEMMA OUTPUT:
['Anxiety Disorders', 'anxiety disorder (a condition corresponding most closely to the APA Diagnostic and Statistical Manual [DSM III R] diagnosis of generalized anxiety disorder)', 'short term relief of symptoms of Anxiety', 'Anxiety or tension associated with the stress of everyday life', 'Generalized anxiety disorder', 'Unrealistic or excessive Anxiety and worry (apprehensive expectation)', 'Motor Tension', 'Autonomic Hyperactivity', 'Vigilance and Scanning', 'Anxiety associated with depression', 'Panic Disorder', 'Panic Disorder (DSM IV)', 'recurrent unexpected panic attacks', 'palpitations, pounding heart, or accelerated

In [17]:
for i in range(5):

    print("=" * 80)
    print("RECORD ID:", df.loc[i, "record_id"])

    print()
    print("STATUS:")
    print(df.loc[i, "medgemma_status"])

    print()
    print("RAW RESPONSE:")
    print(repr(df.loc[i, "medgemma_raw"]))

    print()

RECORD ID: 10004

STATUS:
FORMAT_FAILED

RAW RESPONSE:
'<unused94> thought\nThe user wants me to extract all diseases, disorders, syndromes, symptoms, or clinical conditions mentioned in the "Indications and Usage" text for which the drug (Pantoprazole sodium delayed release tablets) is explicitly indicated.\n\n1.  **Identify the drug:** Pantoprazole sodium delayed release tablets.\n2.  **Scan the text for indications:**\n    *   "Short Term Treatment of Erosive Esophagitis Associated With Gastroesophageal Reflux Disease ( GERD )" -> Erosive Esophagitis, GERD\n    *   "short term treatment (up to 8 weeks) in the healing and symptomatic relief of Erosive Esophagitis" -> Erosive Esophagitis (healing and symptomatic relief)\n    *   "maintenance of healing of Erosive Esophagitis and reduction in relapse rates of daytime and nighttime heartburn symptoms in adult patients with GERD" -> Erosive Esophagitis (maintenance of healing), heartburn symptoms (relapse rates)\n    *   "pathological hy

In [18]:
test_text = """
Metoprolol is indicated for the treatment of hypertension and angina pectoris.
"""

diseases, raw, runtime, status = extract_medgemma(
    test_text
)

print("=" * 70)
print("MEDGEMMA SIMPLE TEST")
print("=" * 70)

print("\nRAW RESPONSE:")
print(repr(raw))

print("\nPARSED DISEASES:")
print(diseases)

print("\nSTATUS:")
print(status)

print("\nRUNTIME:")
print(runtime, "seconds")

MEDGEMMA SIMPLE TEST

RAW RESPONSE:
'<unused94> thought\nThe user wants me to extract all diseases, disorders, syndromes, symptoms, or clinical conditions mentioned in the "Indications and Usage" section of the provided text.\n\n1.  **Read the text:** "Metoprolol is indicated for the treatment of hypertension and angina pectoris."\n2.  **Identify potential conditions:** "hypertension" and "angina pectoris".\n3.  **Check against rules:**\n    *   Are these explicitly stated? Yes.\n    *   Are they diseases/disorders/syndromes/symptoms/clinical conditions? Yes.\n    *   Are there subtypes/modifiers? No, these are the primary conditions.\n    *   Is the wording close to the text? Yes.\n    *   Are they normalized/generalized? No.\n    *   Are they inferred? No.\n    *   Are conditions explicitly excluded? No.\n    *   Are causative organisms excluded? No.\n    *   Are mechanisms/procedures/dosing/regimens/demographics excluded? Yes.\n    *   Are duplicates removed? No duplicates found.\n4

In [19]:
import json
import re


def parse_output(raw_output):

    raw_output = str(raw_output).strip()

    parsed = None


    # ----------------------------------------------------------
    # 1. MedGemma often puts the final answer inside:
    #
    # ```json
    # {"diseases":[...]}
    # ```
    #
    # Take the LAST JSON block because reasoning may appear first.
    # ----------------------------------------------------------

    json_blocks = re.findall(
        r"```json\s*(\{.*?\})\s*```",
        raw_output,
        flags=re.DOTALL | re.IGNORECASE
    )


    if json_blocks:

        json_text = json_blocks[-1]

        try:
            parsed = json.loads(
                json_text
            )

        except json.JSONDecodeError:
            parsed = None


    # ----------------------------------------------------------
    # 2. Try parsing the whole response
    # ----------------------------------------------------------

    if parsed is None:

        try:

            parsed = json.loads(
                raw_output
            )

        except json.JSONDecodeError:

            parsed = None


    # ----------------------------------------------------------
    # 3. Last fallback:
    # Find the LAST {"diseases"...} JSON object
    # ----------------------------------------------------------

    if parsed is None:

        start = raw_output.rfind(
            '{"diseases"'
        )


        if start == -1:

            raise ValueError(
                "No diseases JSON object found."
            )


        candidate = raw_output[
            start:
        ]


        decoder = json.JSONDecoder()


        try:

            parsed, end_position = (
                decoder.raw_decode(
                    candidate
                )
            )

        except json.JSONDecodeError:

            raise ValueError(
                "Diseases JSON found but malformed."
            )


    # ----------------------------------------------------------
    # 4. Validate expected structure
    # ----------------------------------------------------------

    if "diseases" not in parsed:

        raise ValueError(
            "JSON does not contain 'diseases'."
        )


    diseases = parsed[
        "diseases"
    ]


    if not isinstance(
        diseases,
        list
    ):

        raise ValueError(
            "'diseases' must be a list."
        )


    # ----------------------------------------------------------
    # 5. Clean duplicates only
    # ----------------------------------------------------------

    cleaned = []

    seen = set()


    for disease in diseases:

        if not isinstance(
            disease,
            str
        ):
            continue


        disease = disease.strip()


        if not disease:
            continue


        key = disease.lower()


        if key not in seen:

            seen.add(
                key
            )

            cleaned.append(
                disease
            )


    return cleaned

In [20]:
for i in range(5):

    record_id = df.loc[
        i,
        "record_id"
    ]


    indication_text = str(
        df.loc[
            i,
            "indication_text"
        ]
    )


    print()
    print("=" * 70)

    print(
        f"RECORD {i + 1}/5"
    )

    print(
        "Record ID:",
        record_id
    )


    print()
    print(
        "EXPERT GOLD:"
    )

    print(
        df.loc[
            i,
            "gold_expert_mentions"
        ]
    )


    try:

        diseases, raw, runtime, status = (
            extract_medgemma(
                indication_text
            )
        )


        df.at[
            i,
            "medgemma_output"
        ] = json.dumps(
            diseases,
            ensure_ascii=False
        )


        df.at[
            i,
            "medgemma_raw"
        ] = raw


        df.at[
            i,
            "medgemma_status"
        ] = status


        df.at[
            i,
            "medgemma_runtime_sec"
        ] = runtime


        print()
        print(
            "MEDGEMMA OUTPUT:"
        )

        print(
            diseases
        )


        print(
            "Status:",
            status
        )


        print(
            "Runtime:",
            runtime,
            "seconds"
        )


    except Exception as e:

        df.at[
            i,
            "medgemma_status"
        ] = "FAILED"


        df.at[
            i,
            "medgemma_raw"
        ] = str(e)


        print()
        print(
            "TECHNICAL ERROR:"
        )

        print(
            e
        )


# Save the updated 5-record test
df.to_excel(
    RESULT_FILE,
    index=False
)


print()
print(
    "Saved:"
)

print(
    RESULT_FILE
)


RECORD 1/5
Record ID: 10004

EXPERT GOLD:
Erosive Esophagitis; Gastroesophageal Reflux Disease; GERD; Zollinger Ellison Syndrome

MEDGEMMA OUTPUT:
['Erosive Esophagitis', 'GERD', 'heartburn symptoms', 'pathological hypersecretory conditions', 'Zollinger Ellison Syndrome']
Status: OK
Runtime: 17.34 seconds

RECORD 2/5
Record ID: 10008

EXPERT GOLD:
Anxiety Disorders; anxiety disorder; Anxiety; Anxiety associated with depression; Panic Disorder; agoraphobia

MEDGEMMA OUTPUT:
['Anxiety Disorders', 'anxiety disorder', 'Generalized anxiety disorder', 'Anxiety', 'Panic Disorder', 'Panic Disorder (DSM IV)']
Status: OK
Runtime: 22.48 seconds

RECORD 3/5
Record ID: 10012

EXPERT GOLD:
ventricular fibrillation; ventricular tachycardia; VT; VF

MEDGEMMA OUTPUT:
['ventricular fibrillation', 'hemodynamically unstable ventricular tachycardia', 'ventricular tachycardia', 'ventricular arrhythmias']
Status: OK
Runtime: 12.78 seconds

RECORD 4/5
Record ID: 10016

EXPERT GOLD:
dermatoses; plaque type ps

In [21]:
i = df.index[
    df["record_id"] == 10008
][0]

print("=" * 80)
print("RECORD ID:", df.at[i, "record_id"])

print("\nRAW MEDGEMMA RESPONSE:")
print(repr(df.at[i, "medgemma_raw"]))

RECORD ID: 10008

RAW MEDGEMMA RESPONSE:
'<unused94> thought\nThe user wants me to extract all diseases, disorders, syndromes, symptoms, or clinical conditions mentioned in the "INDICATIONS AND USAGE" text for which the drug (Alprazolam) is explicitly indicated.\n\n1.  **Identify the drug:** Alprazolam tablets.\n2.  **Scan the text for indications:** Look for phrases like "indicated for", "treatment of", "management of", "relief of symptoms of".\n3.  **Extract conditions explicitly mentioned:**\n    *   "Anxiety Disorders" - This is a general category.\n    *   "anxiety disorder" (DSM III R diagnosis of generalized anxiety disorder) - This is a specific condition.\n    *   "short term relief of symptoms of Anxiety" - This refers to anxiety symptoms, not a specific disorder.\n    *   "Anxiety or tension associated with the stress of everyday life" - This is a description of a state, not a specific disorder.\n    *   "Generalized anxiety disorder" - This is a specific condition.\n    *  

In [22]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

try:
    response = requests.get(
        "http://127.0.0.1:11434/api/tags",
        timeout=10
    )

    print("Ollama status:", response.status_code)

except Exception as e:
    print("Ollama still unavailable:", e)

Ollama status: 200


In [23]:
!ollama list

NAME              ID              SIZE      MODIFIED      
medgemma1.5:4b    433252621ab1    3.3 GB    5 minutes ago    


In [24]:
!ollama pull medgemma1.5:4b

In [25]:
!ollama ps

NAME              ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
medgemma1.5:4b    433252621ab1    2.9 GB    100% GPU     4096       4 minutes from now    


In [26]:
i = df.index[
    df["record_id"] == 10008
][0]

indication_text = str(
    df.at[i, "indication_text"]
)

diseases, raw, runtime, status = extract_medgemma(
    indication_text
)

print("RAW:")
print(raw)

print("\nDISEASES:")
print(diseases)

print("\nSTATUS:")
print(status)

print("\nRUNTIME:")
print(runtime)

RAW:
<unused94>thought
The user wants me to extract all diseases, disorders, syndromes, symptoms, or clinical conditions mentioned in the "INDICATIONS AND USAGE" text for which the drug (Alprazolam) is explicitly indicated.

1.  **Identify the drug:** Alprazolam tablets.
2.  **Scan the text for indications:** Look for phrases like "indicated for", "treatment of", "management of", "relief of symptoms of".
3.  **Extract conditions associated with Alprazolam:**
    *   "Anxiety Disorders" - Alprazolam is indicated for the management of anxiety disorder.
    *   "Anxiety" - Alprazolam is indicated for the short term relief of symptoms of Anxiety.
    *   "Generalized anxiety disorder" - Mentioned as a corresponding condition for Alprazolam indication for anxiety disorder.
    *   "Anxiety or tension associated with the stress of everyday life" - Mentioned as a condition where treatment might be needed, implying it's a clinical condition/symptom complex.
    *   "Anxiety" (again) - Mentione

In [27]:
df.at[i, "medgemma_output"] = json.dumps(
    diseases,
    ensure_ascii=False
)

df.at[i, "medgemma_raw"] = raw

df.at[i, "medgemma_status"] = status

df.at[i, "medgemma_runtime_sec"] = runtime

df.to_excel(
    RESULT_FILE,
    index=False
)

print("Saved.")

Saved.


In [28]:
!tail -50 /content/ollama.log

srv        update:    - prompt 0x3c9d1a10:    1026 tokens, checkpoints:  4,   236.670 MiB
srv        update:    - prompt 0x3c9d18b0:    1162 tokens, checkpoints:  4,   243.860 MiB
srv        update:    - prompt 0x3c9d1960:    3333 tokens, checkpoints:  4,   262.269 MiB
srv  get_availabl: prompt cache update took 256.01 ms
slot launch_slot_: id  0 | task -1 | sampler chain: logits -> ?penalties -> ?dry -> ?top-n-sigma -> top-k -> ?typical -> top-p -> ?min-p -> ?xtc -> temp-ext -> dist 
slot launch_slot_: id  0 | task -1 | sampler params: 
	repeat_last_n = 64, repeat_penalty = 1.000, frequency_penalty = 0.000, presence_penalty = 0.000
	dry_multiplier = 0.000, dry_base = 1.750, dry_allowed_length = 2, dry_penalty_last_n = 64
	top_k = 40, top_p = 0.900, min_p = 0.000, xtc_probability = 0.000, xtc_threshold = 0.100, typical_p = 1.000, top_n_sigma = -1.000, temp = 0.100
	mirostat = 0, mirostat_lr = 0.100, mirostat_ent = 5.000, adaptive_target = -1.000, adaptive_decay = 0.900
slot launch_slot

In [29]:
i = df.index[
    df["record_id"] == 10008
][0]

indication_text = str(
    df.at[i, "indication_text"]
)

diseases, raw, runtime, status = extract_medgemma(
    indication_text
)

print("DISEASES:")
print(diseases)

print("\nSTATUS:")
print(status)

print("\nRUNTIME:")
print(runtime)

print("\nRAW END:")
print(raw[-1500:])

DISEASES:
['Anxiety Disorders', 'anxiety disorder', 'Anxiety', 'tension associated with the stress of everyday life', 'Generalized anxiety disorder', 'Unrealistic or excessive Anxiety', 'worry (apprehensive expectation)', 'Motor Tension', 'Autonomic Hyperactivity', 'Vigilance and Scanning', 'Anxiety associated with depression', 'Panic Disorder', 'Panic Disorder (DSM IV)', 'recurrent unexpected panic attacks', 'palpitations, pounding heart, or accelerated heart rate', 'sweating', 'trembling or shaking', 'sensations of shortness of breath or smothering', 'feeling of choking', 'chest pain or discomfort', 'nausea or abdominal distress', 'feeling dizzy, unsteady, lightheaded, or faint', 'derealization (feelings of unreality)', 'depersonalization (being detached from oneself)', 'fear of losing control', 'fear of dying', 'paresthesias (numbness or tingling sensations)', 'chills or hot flushes']

STATUS:
OK

RUNTIME:
35.22

RAW END:
   sweating
    *   trembling or shaking
    *   sensations o

In [30]:
num_ctx = 4096
num_predict = 3000
temperature = 0.1

In [31]:
def run_all_medgemma():

    total = len(df)

    for i in range(total):

        record_id = df.loc[i, "record_id"]

        current_status = str(
            df.at[i, "medgemma_status"]
        ).strip()


        # ======================================================
        # Skip records already completed
        # ======================================================

        if current_status in [
            "OK",
            "OK_EMPTY",
            "FORMAT_FAILED",
            "TRUNCATED"
        ]:

            print(
                f"[{i + 1}/{total}] "
                f"Record {record_id} - already completed"
            )

            continue


        indication_text = str(
            df.loc[i, "indication_text"]
        )


        print()
        print("=" * 70)

        print(
            f"[{i + 1}/{total}] "
            f"Record ID: {record_id}"
        )


        success = False
        last_error = ""


        # ======================================================
        # Retry only genuine technical/API failures
        # ======================================================

        for attempt in range(1, 4):

            try:

                diseases, raw, runtime, status = (
                    extract_medgemma(
                        indication_text
                    )
                )


                # ----------------------------------------------
                # Save model prediction
                # ----------------------------------------------

                df.at[
                    i,
                    "medgemma_output"
                ] = json.dumps(
                    diseases,
                    ensure_ascii=False
                )


                df.at[
                    i,
                    "medgemma_raw"
                ] = raw


                df.at[
                    i,
                    "medgemma_status"
                ] = status


                df.at[
                    i,
                    "medgemma_runtime_sec"
                ] = runtime


                print(
                    "Output:",
                    diseases
                )

                print(
                    "Status:",
                    status
                )

                print(
                    "Runtime:",
                    runtime,
                    "seconds"
                )


                success = True
                break


            except Exception as e:

                last_error = str(e)

                print(
                    f"Technical attempt "
                    f"{attempt}/3 failed:"
                )

                print(e)

                time.sleep(3)


        # ======================================================
        # Real technical failure after 3 attempts
        # ======================================================

        if not success:

            df.at[
                i,
                "medgemma_status"
            ] = "FAILED"


            df.at[
                i,
                "medgemma_raw"
            ] = last_error


            print(
                "Final technical status: FAILED"
            )


        # ======================================================
        # Save checkpoint every 20 records
        # ======================================================

        if (
            (i + 1) % 20 == 0
            or
            i == total - 1
        ):

            df.to_excel(
                RESULT_FILE,
                index=False
            )

            print(
                ">>> CHECKPOINT SAVED TO GOOGLE DRIVE"
            )


    # ==========================================================
    # Final save
    # ==========================================================

    df.to_excel(
        RESULT_FILE,
        index=False
    )


    print()
    print("=" * 70)
    print("MEDGEMMA FINISHED")
    print("=" * 70)

    print(
        "Results saved to:"
    )

    print(
        RESULT_FILE
    )

In [32]:
run_all_medgemma()

[1/500] Record 10004 - already completed
[2/500] Record 10008 - already completed
[3/500] Record 10012 - already completed
[4/500] Record 10016 - already completed
[5/500] Record 10017 - already completed

[6/500] Record ID: 1001
Output: ['Hypertension', 'Angina Pectoris', 'Myocardial Infarction']
Status: OK
Runtime: 10.58 seconds

[7/500] Record ID: 10140
Output: ['major depressive disorder', 'depression', 'Major Depression']
Status: OK
Runtime: 15.92 seconds

[8/500] Record ID: 10146
Output: ['insomnia']
Status: OK
Runtime: 7.07 seconds

[9/500] Record ID: 10153
Output: ['Anxiety', 'Tension', 'Psychoneurosis', 'Pruritus', 'Allergic conditions', 'Chronic urticaria', 'Atopic dermatoses', 'Contact dermatoses', 'Histamine mediated pruritus']
Status: OK
Runtime: 25.46 seconds

[10/500] Record ID: 10157
Output: ["Parkinson's disease", "idiopathic Parkinson's disease", "early Parkinson's disease", 'advanced disease']
Status: OK
Runtime: 13.96 seconds

[11/500] Record ID: 10173
Output: ['par

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
!apt-get update -qq
!apt-get install -y zstd pciutils
!pip -q install pandas openpyxl requests

!curl -fsSL https://ollama.com/install.sh | sh

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
pciutils is already the newest version (1:3.10.0-2build1).
0 upgraded, 0 newly installed, 0 to remove and 85 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama sys

In [35]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

response = requests.get(
    "http://127.0.0.1:11434/api/tags"
)

print("Ollama status:", response.status_code)

Ollama status: 200


In [36]:
!ollama list

NAME              ID              SIZE      MODIFIED    
medgemma1.5:4b    433252621ab1    3.3 GB    3 hours ago    


In [37]:
!ollama pull medgemma1.5:4b

In [38]:
from pathlib import Path
import pandas as pd

RESULT_FILE = Path(
    "/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_MedGemma_Results.xlsx"
)

print("Existing result file:", RESULT_FILE.exists())

df = pd.read_excel(
    RESULT_FILE
)

print("Records:", len(df))

Existing result file: True
Records: 500


In [39]:
print(
    df["medgemma_status"]
    .value_counts(dropna=False)
)

medgemma_status
OK               480
FORMAT_FAILED     19
OK_EMPTY           1
Name: count, dtype: int64


In [40]:
completed_statuses = {
    "OK",
    "OK_EMPTY",
    "FORMAT_FAILED",
    "TRUNCATED",
    "OK_REPAIRED",
    "FORMAT_FAILED_TRUNCATED"
}

status_clean = (
    df["medgemma_status"]
    .fillna("")
    .astype(str)
    .str.strip()
)

pending = df.index[
    ~status_clean.isin(completed_statuses)
]

print("Completed records:", len(df) - len(pending))
print("Remaining records:", len(pending))

if len(pending) > 0:

    first_pending = pending[0]

    print(
        "Next row:",
        first_pending + 1
    )

    print(
        "Next record ID:",
        df.at[first_pending, "record_id"]
    )

Completed records: 500
Remaining records: 0


In [41]:
MODEL = "medgemma1.5:4b"

In [42]:
run_all_medgemma()

[1/500] Record 10004 - already completed
[2/500] Record 10008 - already completed
[3/500] Record 10012 - already completed
[4/500] Record 10016 - already completed
[5/500] Record 10017 - already completed
[6/500] Record 1001 - already completed
[7/500] Record 10140 - already completed
[8/500] Record 10146 - already completed
[9/500] Record 10153 - already completed
[10/500] Record 10157 - already completed
[11/500] Record 10173 - already completed
[12/500] Record 1024 - already completed
[13/500] Record 10263 - already completed
[14/500] Record 10268 - already completed
[15/500] Record 10295 - already completed
[16/500] Record 10303 - already completed
[17/500] Record 10311 - already completed
[18/500] Record 10312 - already completed
[19/500] Record 10316 - already completed
[20/500] Record 10318 - already completed
[21/500] Record 10319 - already completed
[22/500] Record 10320 - already completed
[23/500] Record 10322 - already completed
[24/500] Record 10445 - already completed
[25

In [43]:
import json
import re
import time
import requests


MODEL = "medgemma1.5:4b"

URL = "http://127.0.0.1:11434/api/chat"


SYSTEM_MESSAGE = """
You are a biomedical information extraction system.
Follow the user's extraction instructions exactly.
Return only the requested JSON output.
""".strip()


EXTRACTION_TEMPLATE = """
Extract ALL diseases, disorders, syndromes, symptoms, or clinical conditions
for which the drug is explicitly indicated in the following
Indications and Usage text.

Rules:

1. Read the entire supplied text.
2. Include every explicitly stated clinical indication.
3. Include symptoms or clinical conditions when explicitly indicated
   for treatment, prevention, management, or relief.
4. Preserve clinically meaningful disease subtypes and modifiers.
5. Use wording from the supplied text as closely as possible.
6. Do not intentionally normalize or generalize disease names.
7. Do not infer conditions that are not explicitly supported by the text.
8. Exclude conditions explicitly stated as NOT indicated or NOT recommended.
9. Exclude causative organisms when they merely describe the cause of infection.
10. Exclude mechanisms, procedures, dosing information,
    treatment regimens, and patient demographics.
11. Remove duplicate clinical conditions.

Return ONLY valid JSON:

{{"diseases":["condition 1","condition 2"]}}

If there are genuinely no indications:

{{"diseases":[]}}

INDICATIONS AND USAGE:

{indication_text}
""".strip()


def parse_output(raw_output):

    raw_output = str(raw_output).strip()

    parsed = None


    # MedGemma usually places final answer in a fenced JSON block
    json_blocks = re.findall(
        r"```json\s*(\{.*?\})\s*```",
        raw_output,
        flags=re.DOTALL | re.IGNORECASE
    )


    if json_blocks:

        try:
            parsed = json.loads(
                json_blocks[-1]
            )

        except json.JSONDecodeError:
            parsed = None


    # Try whole response
    if parsed is None:

        try:
            parsed = json.loads(
                raw_output
            )

        except json.JSONDecodeError:
            parsed = None


    # Last fallback:
    # locate final {"diseases": ...} object
    if parsed is None:

        start = raw_output.rfind(
            '{"diseases"'
        )

        if start == -1:

            raise ValueError(
                "No diseases JSON object found."
            )


        candidate = raw_output[start:]

        decoder = json.JSONDecoder()


        try:

            parsed, _ = decoder.raw_decode(
                candidate
            )

        except json.JSONDecodeError:

            raise ValueError(
                "Diseases JSON found but malformed."
            )


    if "diseases" not in parsed:

        raise ValueError(
            "JSON does not contain 'diseases'."
        )


    diseases = parsed["diseases"]


    if not isinstance(diseases, list):

        raise ValueError(
            "'diseases' must be a list."
        )


    cleaned = []
    seen = set()


    for disease in diseases:

        if not isinstance(disease, str):
            continue

        disease = disease.strip()

        if not disease:
            continue

        key = disease.lower()

        if key not in seen:

            seen.add(key)

            cleaned.append(
                disease
            )


    return cleaned


def extract_medgemma(indication_text):

    user_message = EXTRACTION_TEMPLATE.format(
        indication_text=indication_text
    )


    payload = {

        "model": MODEL,

        "messages": [

            {
                "role": "system",
                "content": SYSTEM_MESSAGE
            },

            {
                "role": "user",
                "content": user_message
            }
        ],

        "stream": False,

        "options": {

            "temperature": 0.1,

            "num_ctx": 4096,

            # MedGemma needs extra room for reasoning
            "num_predict": 3000
        }
    }


    start_time = time.time()


    response = requests.post(
        URL,
        json=payload,
        timeout=240
    )


    response.raise_for_status()


    raw = (
        response.json()
        ["message"]
        ["content"]
    )


    runtime = round(
        time.time() - start_time,
        2
    )


    try:

        diseases = parse_output(raw)

        status = (
            "OK"
            if diseases
            else "OK_EMPTY"
        )


    except Exception:

        diseases = []

        # MedGemma began reasoning but never reached final answer
        if (
            "<unused94>" in raw
            and
            "<unused95>" not in raw
        ):

            status = "TRUNCATED"

        else:

            status = "FORMAT_FAILED"


    return diseases, raw, runtime, status

In [44]:
print(extract_medgemma)

<function extract_medgemma at 0x79a12edf7d80>


In [45]:
i = df.index[
    df["record_id"] == 12504
][0]

text = str(
    df.at[i, "indication_text"]
)

diseases, raw, runtime, status = extract_medgemma(
    text
)

print("Diseases:", diseases)
print("Status:", status)
print("Runtime:", runtime)

Diseases: ['exogenous obesity']
Status: OK
Runtime: 14.26


In [46]:
!ollama --version

ollama version is 0.34.1


In [ ]:
!apt-get update -qq
!apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
pciutils is already the newest version (1:3.10.0-2build1).
0 upgraded, 0 newly installed, 0 to remove and 85 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
##########################################                                59.5%

In [ ]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

try:
    response = requests.get(
        "http://127.0.0.1:11434/api/tags",
        timeout=10
    )

    print("Ollama status:", response.status_code)

except Exception as e:
    print("Ollama connection error:", e)

In [ ]:
!ollama list

In [ ]:
import requests

r = requests.get(
    "http://127.0.0.1:11434/api/tags",
    timeout=10
)

print(r.status_code)
print(r.json())

In [ ]:
print(extract_medgemma)

In [ ]:
i = df.index[
    df["record_id"] == 12504
][0]

text = str(
    df.at[i, "indication_text"]
)

diseases, raw, runtime, status = extract_medgemma(
    text
)

print("Diseases:", diseases)
print("Status:", status)
print("Runtime:", runtime)

In [ ]:
!nvidia-smi

In [ ]:
import torch
import shutil

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("nvidia-smi path:", shutil.which("nvidia-smi"))

!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

RESULT_FILE = Path(
    "/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_MedGemma_Results.xlsx"
)

df = pd.read_excel(RESULT_FILE)

print("Result file exists:", RESULT_FILE.exists())
print("Records:", len(df))

print()
print(df["medgemma_status"].value_counts(dropna=False))

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))